<a href="https://colab.research.google.com/github/AdwaaNasser/Saudi-Real-Estate-Market-Analysis/blob/main/03_data_splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
!git clone https://github.com/AdwaaNasser/Saudi-Real-Estate-Market-Analysis.git /content/real-estate
%cd /content/real-estate/notebooks
%pip install -r ../requirements.txt



fatal: destination path '/content/real-estate' already exists and is not an empty directory.
/content/real-estate/notebooks


In [38]:

import pandas as pd

df = pd.read_csv('../data/cleaned/clean.csv')
print(df.shape)
df.head()


(34279, 11)


,المنطقة,المدينة,المدينة / الحي,الرقم المرجعي للصفقة,تاريخ الصفقة ميلادي,تاريخ الصفقة هجري,تصنيف العقار,عدد العقارات,السعر,المساحة,سعر_المتر
0,منطقة مكة المكرمه,الطائف,الطائف/ الكدى,35983585,2026/04/01,1447/10/13,سكني,1,1000000.0,593.75,1684.210526
1,منطقة الرياض,حوطة بني تميم,حوطة بني تميم/ الورود,35979045,2026/04/01,1447/10/13,سكني,1,1115626.0,575.00,1940.219130
2,منطقة مكة المكرمه,الطائف,الطائف/ الحلقة الشرقية,35986850,2026/04/01,1447/10/13,سكني,1,600000.0,307.74,1949.697797
3,منطقة مكة المكرمه,الطائف,الطائف/ اليمانيه,35984449,2026/04/01,1447/10/13,سكني,1,245000.0,179.79,1362.700929
4,منطقة الرياض,الدلم,الدلم/ الفيصلية,35972886,2026/04/01,1447/10/13,سكني,1,120000.0,500.00,240.000000


In [39]:
from sklearn.preprocessing import StandardScaler
columns_to_scale = ['السعر', 'المساحة', 'سعر_المتر']

#  تطبيق StandardScaler
scaler = StandardScaler()
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

print(df[columns_to_scale])


          السعر   المساحة  سعر_المتر
0      0.008075 -0.058854  -0.008507
1      0.018893 -0.058992   0.086628
2     -0.029347 -0.060949   0.090150
3     -0.062560 -0.061886  -0.127981
4     -0.074254 -0.059541  -0.545183
...         ...       ...        ...
34274 -0.077997 -0.056503  -0.601870
34275 -0.069109 -0.059911  -0.489681
34276 -0.068641 -0.060002  -0.481339
34277 -0.072851 -0.059945  -0.521586
34278 -0.040226 -0.058346  -0.363295

[34279 rows x 3 columns]


In [40]:
from sklearn.model_selection import train_test_split

# الهدف اللي نبي نتوقعه
y = df['السعر']

# الـfeatures: نشيل السعر، وسعر المتر (لأنه محسوب من السعر = غش)، والرقم المرجعي (مجرد ID)
X = df.drop(columns=['السعر', 'سعر_المتر', 'الرقم المرجعي للصفقة'], errors='ignore')

# التقسيم: 80% تدريب و20% اختبار
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train:', X_train.shape)
print('Test :', X_test.shape)
print('Features:', X.columns.tolist())


Train: (27423, 8)
Test : (6856, 8)
Features: ['المنطقة', 'المدينة', 'المدينة / الحي', 'تاريخ الصفقة ميلادي', 'تاريخ الصفقة هجري', 'تصنيف العقار', 'عدد العقارات', 'المساحة']


In [41]:
import numpy as np
df['target_log'] = np.log1p(df['السعر'])
cols_to_drop =  ['الرقم المرجعي للصفقه', 'تاريخ االصفقة ميلادي', 'تاريخ الصفقه هجري', 'السعر']
x_selected = df.drop(
    columns=[c for c in cols_to_drop if c in df.columns]
)
y_selected = df['target_log']

print('target log-transform and leakage column dropped')

target log-transform and leakage column dropped


In [42]:
df.describe()

,الرقم المرجعي للصفقة,عدد العقارات,السعر,المساحة,سعر_المتر,target_log
count,3.427900e+04,34279.000000,3.427900e+04,3.427900e+04,3.427900e+04,34279.000000
mean,3.689563e+07,1.014849,-2.072822e-18,-4.560209e-18,1.989909e-17,-0.024488
std,5.572545e+05,0.304939,1.000015e+00,1.000015e+00,1.000015e+00,0.136705
min,3.430452e+07,1.000000,-8.501325e-02,-6.320126e-02,-6.343672e-01,-0.088846
25%,3.642049e+07,1.000000,-7.542371e-02,-6.130846e-02,-5.596728e-01,-0.078420
50%,3.685494e+07,1.000000,-4.992956e-02,-5.997081e-02,-3.185032e-01,-0.051219
75%,3.740896e+07,1.000000,-1.718479e-02,-5.839338e-02,3.113913e-01,-0.017334
max,3.790381e+07,18.000000,1.178203e+02,6.492385e+01,3.033274e+01,4.777612


In [43]:
from sklearn.preprocessing import OneHotEncoder

cols = ["المنطقة", "المدينة / الحي", "تصنيف العقار"]

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# تحويل الأعمدة المحددة إلى مصفوفة أرقام (0 و 1)
encoded_array = encoder.fit_transform(df[cols])

# تحويل المصفوفة الناتجة إلى DataFrame مع إعطائها أسماء الأعمدة المناسبة
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(cols))

# دمجها مع باقي أعمدة df بعد حذف الأعمدة النصية القديمة
df_final = pd.concat([df.drop(columns=cols), encoded_df], axis=1)

In [44]:
df_final.head()

,المدينة,الرقم المرجعي للصفقة,تاريخ الصفقة ميلادي,تاريخ الصفقة هجري,عدد العقارات,السعر,المساحة,سعر_المتر,target_log,المنطقة_منطقة الباحة,...,المدينة / الحي_وادي الفرع/ وادي الفرع,المدينة / الحي_وادي الفرع/أخرى,المدينة / الحي_يدمه/ اللجام,المدينة / الحي_يدمه/ شرق محافظة يدمه ب 32 كم,المدينة / الحي_يدمه/ هجر وادي الحبط العزيزية,المدينة / الحي_يدمه/أخرى,المدينة / الحي_ينبع/ السلام مخطط منح هاء 13,تصنيف العقار_تجاري,تصنيف العقار_زراعي,تصنيف العقار_سكني
0,الطائف,35983585,2026/04/01,1447/10/13,1,0.008075,-0.058854,-0.008507,0.008043,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,حوطة بني تميم,35979045,2026/04/01,1447/10/13,1,0.018893,-0.058992,0.086628,0.018717,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,الطائف,35986850,2026/04/01,1447/10/13,1,-0.029347,-0.060949,0.090150,-0.029786,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,الطائف,35984449,2026/04/01,1447/10/13,1,-0.062560,-0.061886,-0.127981,-0.064602,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,الدلم,35972886,2026/04/01,1447/10/13,1,-0.074254,-0.059541,-0.545183,-0.077156,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
